# 01 — Calibration
Parse KITTI calibration files and understand the math that bridges LiDAR and camera coordinate frames.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install numpy matplotlib -q

In [ ]:
import os
import numpy as np

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')
LIDAR_PATH = os.path.join(BASE_PATH, 'velodyne/training/velodyne')
CALIB_PATH = os.path.join(BASE_PATH, 'calib/data_object_calib/training/calib')
LABEL_PATH = os.path.join(BASE_PATH, 'label_2/training/label_2')

sample_id = '000000'
print(os.path.exists(f'{IMAGE_PATH}/{sample_id}.png'))
print(os.path.exists(f'{LIDAR_PATH}/{sample_id}.bin'))
print(os.path.exists(f'{CALIB_PATH}/{sample_id}.txt'))
print(os.path.exists(f'{LABEL_PATH}/{sample_id}.txt'))

In [ ]:
def parse_calib(calib_path):
    calib = {}
    with open(calib_path, 'r') as f:
        for line in f.readlines():
            if ':' in line:
                key, value = line.split(':', 1)
                calib[key.strip()] = np.array(
                    [float(x) for x in value.strip().split()]
                )
    P2 = calib['P2'].reshape(3, 4)
    R0 = calib['R0_rect'].reshape(3, 3)
    Tr = calib['Tr_velo_to_cam'].reshape(3, 4)
    return P2, R0, Tr

first_file = sorted(os.listdir(IMAGE_PATH))[0]
sample_id  = first_file.replace('.png', '')
P2, R0, Tr = parse_calib(f'{CALIB_PATH}/{sample_id}.txt')

print('P2 (camera projection matrix):'); print(P2)
print('R0 (rectification matrix):');     print(R0)
print('Tr (LiDAR to camera transform):');print(Tr)

## What these matrices mean
- **P2** — projects 3D camera-frame points into 2D image pixels. Focal length ~707px, principal point ~(604, 180)
- **R0_rect** — rectification rotation, near-identity meaning cameras are well aligned
- **Tr_velo_to_cam** — rigid body transform from LiDAR frame to camera frame. Last column is physical offset: LiDAR mounted ~33cm above camera